GLOBAL SETUP

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time



In [0]:
SILVER_BASE = "s3://enterprise-lakehouse-data/silver_curated/"
GOLD_BASE   = "s3://enterprise-lakehouse-data/gold/"
TEMP_BASE   = "s3://enterprise-lakehouse-data/_tmp/gold/"
METRICS_PATH = "s3://enterprise-lakehouse-data/metadata/run_metrics/step6_metrics.json"


In [0]:
assets_scd = spark.read.parquet(f"{SILVER_BASE}/assets_scd")
maintenance_scd = spark.read.parquet(f"{SILVER_BASE}/maintenance_events")
reference_scd = spark.read.parquet(f"{SILVER_BASE}/reference_data")


In [0]:
assets_scd.printSchema()

root
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- install_date_dt: date (nullable = true)
 |-- last_updated_dt: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- f_null_asset_id: boolean (nullable = true)
 |-- f_null_asset_type: boolean (nullable = true)
 |-- f_null_status: boolean (nullable = true)
 |-- f_invalid_capacity: boolean (nullable = true)
 |-- f_null_ingestion_ts: boolean (nullable = true)
 |-- failure_reason: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- asset_event_hash: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nulla

In [0]:
maintenance_scd.printSchema()


root
 |-- maintenance_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- actual_date_dt: date (nullable = true)
 |-- maintenance_event_hash: string (nullable = true)
 |-- lateness_days: integer (nullable = true)
 |-- is_late_record: boolean (nullable = true)
 |-- ingestion_date: date (nullable = true)



In [0]:
reference_scd.printSchema()

root
 |-- asset_type: string (nullable = true)
 |-- risk: string (nullable = true)
 |-- cycle_days: integer (nullable = true)
 |-- reference_hash: string (nullable = true)
 |-- ingestion_date: date (nullable = true)



dim_asset_scd

Select ONLY business + SCD columns

In [0]:
asset_base = assets_scd.select(
    "asset_id",
    "plant_id",
    "asset_type",
    "capacity_mw",
    "status",
    "effective_from",
    "effective_to",
    "is_current",
    "asset_event_hash"
)


Reference join (BROADCAST — EXPLICIT)

In [0]:
from pyspark.sql import functions as F

asset_enriched = (
    asset_base
    .join(
        F.broadcast(
            reference_scd.select("asset_type", "risk")
        ),
        on="asset_type",
        how="left"
    )
)


Final Gold dataframe

In [0]:
dim_asset_scd = asset_enriched.select(
    "asset_id",
    "plant_id",
    "asset_type",
    "capacity_mw",
    "status",
    "risk",
    "effective_from",
    "effective_to",
    "is_current",
    "asset_event_hash"
)


GOLD DATA QUALITY (FAIL FAST)






Business key must exist

In [0]:
assert dim_asset_scd.filter(F.col("asset_id").isNull()).count() == 0


One current row per asset (CRITICAL)

In [0]:
current_violation = (
    dim_asset_scd
    .filter(F.col("is_current") == True)
    .groupBy("asset_id")
    .count()
    .filter("count > 1")
)

assert current_violation.count() == 0


No overlapping SCD ranges (VERY IMPORTANT)

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("asset_id").orderBy("effective_from")

overlaps = (
    dim_asset_scd
    .withColumn("prev_effective_to", F.lag("effective_to").over(w))
    .filter(F.col("prev_effective_to").isNotNull())
    .filter(F.col("effective_from") < F.col("prev_effective_to"))
)

assert overlaps.count() == 0


PARTITION STRATEGY (BUSINESS TIME)

In [0]:
dim_asset_scd = (
    dim_asset_scd
    .withColumn("year", F.year("effective_from"))
    .withColumn("month", F.month("effective_from"))
    .withColumn("day", F.dayofmonth("effective_from"))
)


WRITE PATTERN (REPLAY SAFE)

In [0]:
TEMP_PATH  = "s3://enterprise-lakehouse-data/_tmp/gold/dim_asset_scd"
FINAL_PATH = "s3://enterprise-lakehouse-data/gold/dim_asset_scd"


Write to temp

In [0]:
(
    dim_asset_scd
    .repartition(100)   # file size control
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(TEMP_PATH)
)


Validate write

In [0]:
spark.read.parquet(TEMP_PATH).count()


16873

In [0]:
dbutils.fs.rm(FINAL_PATH, recurse=True)
dbutils.fs.mv(TEMP_PATH, FINAL_PATH, recurse=True)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
(
    dim_asset_scd
    .repartition(100)   # file size control
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(FINAL_PATH)
)


In [0]:
assert (
    spark.read.parquet(FINAL_PATH).count()
    == assets_scd.count()
)


In [0]:
spark.read.parquet(FINAL_PATH) \
    .filter("asset_id IS NOT NULL") \
    .orderBy("asset_id", "effective_from") \
    .show(20, truncate=False)


+--------+--------+----------+-----------+--------+------+-------------------+-------------------+----------+----------------------------------------------------------------+----+-----+---+
|asset_id|plant_id|asset_type|capacity_mw|status  |risk  |effective_from     |effective_to       |is_current|asset_event_hash                                                |year|month|day|
+--------+--------+----------+-----------+--------+------+-------------------+-------------------+----------+----------------------------------------------------------------+----+-----+---+
|AST-1000|NULL    |GENERATOR |-100.0     |INACTIVE|HIGH  |2025-03-16 00:00:00|2025-03-25 00:00:00|false     |384b766e6d37154a7a87f050c2ee1edcf62392301b265c410d57f956b0cea83f|2025|3    |16 |
|AST-1000|PLANT-7 |GENERATOR |50.0       |INACTIVE|HIGH  |2025-03-25 00:00:00|2025-04-03 00:00:00|false     |443b0bcafb0d3fea1f316b66956ed38660475cd9b722e8d1f2c0a581bdd1ae24|2025|3    |25 |
|AST-1000|PLANT-10|PUMP      |-50.0      |RETIRED 

GOLD SNAPSHOT DIMENSION



dim_asset_current

In [0]:
dim_asset_scd = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/dim_asset_scd"
)


SNAPSHOT LOGIC (NO HISTORY)

In [0]:
asset_current_base = (
    dim_asset_scd
    .filter(F.col("is_current") == True)
    .filter(F.col("asset_id").isNotNull())
)


FINAL SCHEMA (BI-FRIENDLY, LOCKED)

In [0]:
dim_asset_current = asset_current_base.select(
    "asset_id",
    "asset_type",
    "plant_id",
    "capacity_mw",
    "status",
    "risk",
    F.current_date().alias("last_updated_date")
)


DATA QUALITY CHECKS (MANDATORY)

In [0]:
assert dim_asset_current.filter(F.col("asset_id").isNull()).count() == 0


One row per asset (CRITICAL)

In [0]:
assert (
    dim_asset_current.count()
    == dim_asset_current.select("asset_id").distinct().count()
)


No negative capacity

In [0]:
assert dim_asset_current.filter(F.col("asset_id").isNull()).count() == 0


PARTITIONING (SNAPSHOT DATE)

In [0]:
dim_asset_current = (
    dim_asset_current
    .withColumn("year", F.year("last_updated_date"))
    .withColumn("month", F.month("last_updated_date"))
    .withColumn("day", F.dayofmonth("last_updated_date"))
)


In [0]:
TEMP_PATH  = "s3://enterprise-lakehouse-data/_tmp/gold/dim_asset_current"
FINAL_PATH = "s3://enterprise-lakehouse-data/gold/dim_asset_current"


In [0]:
(
    dim_asset_current
    .repartition(50)   # file size control
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(TEMP_PATH)
)


In [0]:
spark.read.parquet(TEMP_PATH).count()


201

In [0]:
(
    dim_asset_current
    .repartition(50)   # file size control
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(FINAL_PATH)
)


In [0]:
spark.read.parquet(FINAL_PATH).count() \
== dim_asset_scd.filter("is_current = true").count()


True

Duplicate check

In [0]:
spark.read.parquet(FINAL_PATH) \
    .groupBy("asset_id") \
    .count() \
    .filter("count > 1")


DataFrame[asset_id: string, count: bigint]

BI SANITY CHECK

In [0]:
spark.read.parquet(FINAL_PATH) \
    .select("status") \
    .groupBy("status") \
    .count() \
    .show()


+--------+-----+
|  status|count|
+--------+-----+
|INACTIVE|   61|
|  ACTIVE|   46|
| RETIRED|   94|
+--------+-----+



fact_maintenance_daily

In [0]:
maintenance_scd = spark.read.parquet(
    "s3://enterprise-lakehouse-data/silver_curated/maintenance_events"
)


Gold-level business filtering

In [0]:
maintenance_clean = (
    maintenance_scd
    .filter(F.col("asset_id").isNotNull())
    .filter(F.col("actual_date_dt").isNotNull())
    .filter(F.col("cost") >= 0)
)


Aggregate to DAILY GRAIN

In [0]:
fact_maintenance_daily = (
    maintenance_clean
    .groupBy(
        F.col("actual_date_dt").alias("business_date"),
        "asset_id"
    )
    .agg(
        F.sum("cost").alias("total_maintenance_cost"),
        F.count("*").alias("maintenance_count")
    )
)


DATA QUALITY CHECKS (MANDATORY)

No null business keys

In [0]:
assert fact_maintenance_daily.filter(F.col("asset_id").isNull()).count() == 0
assert fact_maintenance_daily.filter(F.col("business_date").isNull()).count() == 0


No negative metrics

In [0]:
assert fact_maintenance_daily.filter(F.col("total_maintenance_cost") < 0).count() == 0


No duplicate rows at grain

In [0]:
assert (
    fact_maintenance_daily.count()
    ==
    fact_maintenance_daily
        .select("asset_id", "business_date")
        .distinct()
        .count()
)


PARTITION STRATEGY (BUSINESS DATE)

In [0]:
fact_maintenance_daily = (
    fact_maintenance_daily
    .withColumn("year", F.year("business_date"))
    .withColumn("month", F.month("business_date"))
    .withColumn("day", F.dayofmonth("business_date"))
)


WRITE PATTERN (ACID-STYLE ON S3)

In [0]:
TEMP_PATH  = "s3://enterprise-lakehouse-data/_tmp/gold/fact_maintenance_daily"
FINAL_PATH = "s3://enterprise-lakehouse-data/gold/fact_maintenance_daily"


In [0]:
(
    fact_maintenance_daily
    .repartition(200)   # controls file size
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(TEMP_PATH)
)


In [0]:
spark.read.parquet(TEMP_PATH).count()


201

In [0]:
(
    fact_maintenance_daily
    .repartition(200)   # controls file size
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(FINAL_PATH)
)


In [0]:
spark.read.parquet(FINAL_PATH).count()


201

In [0]:
spark.read.parquet(FINAL_PATH) \
    .orderBy("business_date") \
    .show(20, truncate=False)


+-------------+--------+----------------------+-----------------+----+-----+---+
|business_date|asset_id|total_maintenance_cost|maintenance_count|year|month|day|
+-------------+--------+----------------------+-----------------+----+-----+---+
|2026-01-07   |AST-1178|66000.0               |5                |2026|1    |7  |
|2026-01-07   |AST-1181|102000.0              |7                |2026|1    |7  |
|2026-01-07   |AST-1172|66000.0               |5                |2026|1    |7  |
|2026-01-07   |AST-1171|59000.0               |5                |2026|1    |7  |
|2026-01-07   |AST-1200|61000.0               |8                |2026|1    |7  |
|2026-01-07   |AST-1173|44000.0               |6                |2026|1    |7  |
|2026-01-07   |AST-1177|104000.0              |6                |2026|1    |7  |
|2026-01-07   |AST-1175|82000.0               |7                |2026|1    |7  |
|2026-01-07   |AST-1184|89000.0               |7                |2026|1    |7  |
|2026-01-07   |AST-1185|4200

In [0]:
maintenance_clean.agg(F.sum("cost")).show()
spark.read.parquet(FINAL_PATH).agg(F.sum("total_maintenance_cost")).show()


+---------+
|sum(cost)|
+---------+
| 1.7524E7|
+---------+

+---------------------------+
|sum(total_maintenance_cost)|
+---------------------------+
|                   1.7524E7|
+---------------------------+



fact_asset_status_daily

Read Gold SCD

In [0]:
dim_asset_scd = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/dim_asset_scd"
)


Gold-level business filtering

In [0]:
asset_scd_clean = (
    dim_asset_scd
    .filter(F.col("asset_id").isNotNull())
    .withColumn("eff_from_date", F.to_date("effective_from"))
    .withColumn("eff_to_date", F.to_date("effective_to"))
)


Expand SCD into DAILY rows (IMPORTANT)

In [0]:
asset_status_expanded = (
    asset_scd_clean
    .withColumn(
        "business_date",
        F.explode(
            F.sequence(
                F.col("eff_from_date"),
                F.least(
                    F.date_sub(F.col("eff_to_date"), 1),
                    TODAY
                )
            )
        )
    )
    .select(
        "asset_id",
        "status",
        "effective_from",
        "business_date"
    )
)


DATA QUALITY CHECKS (MANDATORY)

No null business keys

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy(
        "asset_id",
        "business_date"
    ).orderBy(
        F.col("effective_from").desc()
    )

fact_asset_status_daily = (
    asset_status_expanded
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "business_date",
        "asset_id",
        "status"
    )
)


No duplicate rows at grain

In [0]:
fact_asset_status_daily = (
    fact_asset_status_daily
    .filter(F.col("business_date") <= TODAY)
)


In [0]:
assert fact_asset_status_daily.filter(
    F.col("business_date") > TODAY
).count() == 0


WRITE & FREEZE fact_asset_status_daily (IF NOT DONE)

In [0]:
fact_asset_status_daily = (
    fact_asset_status_daily
    .withColumn("year", F.year("business_date"))
    .withColumn("month", F.month("business_date"))
    .withColumn("day", F.dayofmonth("business_date"))
)


In [0]:
fact_asset_status_daily.count()

59772

ACID-style write

In [0]:
TEMP_PATH  = "s3://enterprise-lakehouse-data/_tmp/gold/fact_asset_status_daily"
FINAL_PATH = "s3://enterprise-lakehouse-data/gold/fact_asset_status_daily"






In [0]:
(
    fact_asset_status_daily
    .repartition(300)
    .write
    
    .partitionBy("year", "month", "day")
    .parquet(TEMP_PATH)
)



In [0]:
(
    fact_asset_status_daily
    .repartition(300)
    .write
    
    .partitionBy("year", "month", "day")
    .parquet(FINAL_PATH)
)


In [0]:
spark.read.parquet(FINAL_PATH).count()

59772

In [0]:
dim_asset_current = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/dim_asset_current"
)

fact_asset_status_daily = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/fact_asset_status_daily"
)

fact_maintenance_daily = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/fact_maintenance_daily"
)


JOIN STRATEGY (VERY IMPORTANT)
Asset status → asset dimension

Left join (never drop facts)

In [0]:
status_with_plant = (
    fact_asset_status_daily
    .join(
        dim_asset_current.select("asset_id", "plant_id", "capacity_mw"),
        on="asset_id",
        how="left"
    )
)


Maintenance → asset dimension

(also left join)

In [0]:
maintenance_with_plant = (
    fact_maintenance_daily
    .join(
        dim_asset_current.select("asset_id", "plant_id"),
        on="asset_id",
        how="left"
    )
)


AGGREGATIONS (CORE LOGIC)

In [0]:
status_kpis = (
    status_with_plant
    .groupBy("plant_id", "business_date")
    .agg(
        F.sum(F.when(F.col("status") == "ACTIVE", 1).otherwise(0))
            .alias("active_assets"),
        F.sum(F.when(F.col("status") != "ACTIVE", 1).otherwise(0))
            .alias("downtime_assets"),
        F.avg("capacity_mw").alias("avg_capacity")
    )
)


Maintenance KPIs

In [0]:
maintenance_kpis = (
    maintenance_with_plant
    .groupBy("plant_id", "business_date")
    .agg(
        F.sum("total_maintenance_cost").alias("total_maintenance_cost")
    )
)


FINAL KPI TABLE (LEFT JOIN)

In [0]:
kpi_plant_daily_summary = (
    status_kpis
    .join(
        maintenance_kpis,
        on=["plant_id", "business_date"],
        how="left"
    )
    .fillna({"total_maintenance_cost": 0.0})
)


DATA QUALITY CHECKS (MANDATORY)

In [0]:
kpi_plant_daily_summary = (
    kpi_plant_daily_summary
    .filter(F.col("plant_id").isNotNull())
)


In [0]:
assert kpi_plant_daily_summary.filter(F.col("plant_id").isNull()).count() == 0
assert kpi_plant_daily_summary.filter(F.col("business_date").isNull()).count() == 0


In [0]:
assert kpi_plant_daily_summary.filter(
    F.col("total_maintenance_cost") < 0
).count() == 0


In [0]:
assert kpi_plant_daily_summary.filter(
    F.col("active_assets") < 0
).count() == 0


PARTITIONING (MANDATORY)

In [0]:
kpi_plant_daily_summary = (
    kpi_plant_daily_summary
    .withColumn("year", F.year("business_date"))
    .withColumn("month", F.month("business_date"))
    .withColumn("day", F.dayofmonth("business_date"))
)


In [0]:
kpi_plant_daily_summary \
    .coalesce(10) \
    .write \
    .mode("overwrite") \
    .partitionBy("year", "month", "day") \
    .parquet("s3://enterprise-lakehouse-data/gold/kpi_plant_daily_summary")


In [0]:
spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/kpi_plant_daily_summary"
).count()


3034